In [2]:
# ============================================================
# Install extra libraries (if not already in the Kaggle image)
# ============================================================
!pip install -q timm lightning torchmetrics

import os
from pathlib import Path
import numpy as np
import pandas as pd

import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

from sklearn.model_selection import KFold

import timm
from lightning.pytorch import LightningModule, Trainer, seed_everything
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from torchmetrics import MeanAbsoluteError


In [3]:
# Root where Kaggle mounted the competition data
# Change "csiro-biomass" to your dataset folder name if needed.
DATA_ROOT = Path("/kaggle/input/csiro-biomass")

TRAIN_CSV = DATA_ROOT / "train.csv"
TEST_CSV = DATA_ROOT / "test.csv"

TRAIN_IMG_ROOT = DATA_ROOT  # image_path already contains "train/..."
TEST_IMG_ROOT = DATA_ROOT   # image_path already contains "test/..."

# Where we write cached npy files and fold checkpoints
WORK_ROOT = Path("/kaggle/working")
CACHE_ROOT = WORK_ROOT / "cache"
FOLD_DIR = WORK_ROOT / "cv_folds"

CACHE_ROOT.mkdir(parents=True, exist_ok=True)
FOLD_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT :", DATA_ROOT)
print("CACHE_ROOT:", CACHE_ROOT)
print("FOLD_DIR  :", FOLD_DIR)

seed_everything(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


PermissionError: [Errno 13] Permission denied: '/kaggle'

In [ ]:
from dataclasses import dataclass
from typing import Optional, List

# ============================================================
# Config
# ============================================================
@dataclass
class Config:
    input_dir: str = str(DATA_ROOT)
    output_dir: str = str(WORK_ROOT / "outputs")

    image_size: tuple[int, int] = (224, 224)

    target_cols: List[str] = None
    target_weights: dict = None

    use_white_balance: bool = True
    use_clahe: bool = True
    use_vegetation_indices: bool = True
    n_vegetation_channels: int = 3  # ExG, VARI, NDI

    def __post_init__(self):
        if self.target_cols is None:
            self.target_cols = [
                "Dry_Green_g",
                "Dry_Dead_g",
                "Dry_Clover_g",
                "GDM_g",
                "Dry_Total_g",
            ]
        if self.target_weights is None:
            self.target_weights = {
                "Dry_Green_g": 0.1,
                "Dry_Dead_g": 0.1,
                "Dry_Clover_g": 0.1,
                "GDM_g": 0.2,
                "Dry_Total_g": 0.5,
            }

    @property
    def total_input_channels(self) -> int:
        n = 3
        if self.use_vegetation_indices:
            n += self.n_vegetation_channels
        return n

    def get_preprocessing_summary(self) -> str:
        enabled = []
        if self.use_white_balance:
            enabled.append("WhiteBalance")
        if self.use_clahe:
            enabled.append("CLAHE")
        if self.use_vegetation_indices:
            enabled.append("VegIdx")
        return ", ".join(enabled) if enabled else "None"


# ============================================================
# Vegetation indices
# ============================================================
class VegetationIndices:
    @staticmethod
    def calculate_exg(img: np.ndarray) -> np.ndarray:
        img = img.astype(np.float32)
        r, g, b = img[:, :, 0], img[:, :, 1], img[:, :, 2]
        exg = 2 * g - r - b
        return np.clip(exg, 0, 255).astype(np.uint8)

    @staticmethod
    def calculate_vari(img: np.ndarray) -> np.ndarray:
        r, g, b = img[:, :, 0], img[:, :, 1], img[:, :, 2]
        denom = g + r - b
        denom = np.where(denom == 0, 1, denom)
        vari = (g - r) / denom
        vari = ((vari + 1.0) * 127.5).astype(np.uint8)
        return vari

    @staticmethod
    def calculate_ndi(img: np.ndarray) -> np.ndarray:
        r, g = img[:, :, 0].astype(np.float32), img[:, :, 1].astype(np.float32)
        denom = r + g
        denom = np.where(denom == 0, 1, denom)
        ndi = (r - g) / denom
        ndi = ((ndi + 1.0) * 127.5).astype(np.uint8)
        return ndi

    @staticmethod
    def calculate_all(img: np.ndarray) -> np.ndarray:
        exg = VegetationIndices.calculate_exg(img)
        vari = VegetationIndices.calculate_vari(img)
        ndi = VegetationIndices.calculate_ndi(img)
        return np.stack([exg, vari, ndi], axis=-1)


# ============================================================
# White balance + CLAHE
# ============================================================
class AdaptivePreprocessor:
    @staticmethod
    def white_balance(img: np.ndarray) -> np.ndarray:
        result = img.astype(np.float32)
        avg_r = np.mean(result[:, :, 0])
        avg_g = np.mean(result[:, :, 1])
        avg_b = np.mean(result[:, :, 2])
        avg_gray = (avg_r + avg_g + avg_b) / 3.0

        if avg_r > 0:
            result[:, :, 0] *= (avg_gray / avg_r)
        if avg_g > 0:
            result[:, :, 1] *= (avg_gray / avg_g)
        if avg_b > 0:
            result[:, :, 2] *= (avg_gray / avg_b)

        return np.clip(result, 0, 255).astype(np.uint8)

    @staticmethod
    def clahe(img: np.ndarray) -> np.ndarray:
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        l2 = clahe.apply(l)
        lab2 = cv2.merge([l2, a, b])
        rgb = cv2.cvtColor(lab2, cv2.COLOR_LAB2RGB)
        return rgb


class ImagePreprocessor:
    @staticmethod
    def preprocess(image: np.ndarray, config: Config) -> np.ndarray:
        if config.use_white_balance:
            image = AdaptivePreprocessor.white_balance(image)
        if config.use_clahe:
            image = AdaptivePreprocessor.clahe(image)

        channels = [image]
        if config.use_vegetation_indices:
            veg = VegetationIndices.calculate_all(image)
            channels.append(veg)
        return np.concatenate(channels, axis=-1)


# ============================================================
# Torchvision transforms
# ============================================================
def get_transforms(config: Config, is_train=True):
    if is_train:
        return transforms.Compose([
            transforms.Resize(config.image_size),
            transforms.RandomHorizontalFlip(0.5),
            transforms.RandomVerticalFlip(0.5),
            transforms.RandomRotation(10),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),
        ])
    else:
        return transforms.Compose([
            transforms.Resize(config.image_size),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),
        ])


In [ ]:
class BiomassDataset(Dataset):
    """
    Online preprocessing dataset (no cache), used mainly as reference.
    On Kaggle we'll usually use the cached version below.
    """
    def __init__(self, df: pd.DataFrame, config: Config,
                 transform: Optional[transforms.Compose] = None):
        self.df = df.reset_index(drop=True)
        self.config = config
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.config.input_dir, row["image_path"])
        img = cv2.imread(img_path)
        if img is None:
            raise FileNotFoundError(f"Image not found: {img_path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        img = ImagePreprocessor.preprocess(img, self.config)

        rgb = img[:, :, :3].astype(np.uint8)
        extra = img[:, :, 3:].astype(np.float32) / 255.0

        rgb_pil = Image.fromarray(rgb)
        if self.transform is not None:
            rgb_tensor = self.transform(rgb_pil)
        else:
            rgb_tensor = transforms.ToTensor()(rgb_pil)

        extra_tensor = torch.from_numpy(extra.transpose(2, 0, 1))
        extra_tensor = torch.nn.functional.interpolate(
            extra_tensor.unsqueeze(0),
            size=self.config.image_size,
            mode="bilinear",
            align_corners=False,
        ).squeeze(0)

        img_tensor = torch.cat([rgb_tensor, extra_tensor], dim=0)

        targets = torch.tensor(
            [np.log1p(row[col]) for col in self.config.target_cols],
            dtype=torch.float32,
        )
        return img_tensor, targets


class CachedBiomassDataset(Dataset):
    """
    Dataset that loads preprocessed images saved as .npy:
      cache_root / (image_path + ".npy")
    where image_path is like "train/ID123.jpg" or "test/ID123.jpg".
    """
    def __init__(self, df: pd.DataFrame, config: Config,
                 cache_root: Path,
                 transform: Optional[transforms.Compose] = None):
        self.df = df.reset_index(drop=True)
        self.config = config
        self.cache_root = cache_root
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        rel_path = row["image_path"]  # e.g. "train/ID123.jpg"
        cache_path = self.cache_root / (rel_path + ".npy")

        if not cache_path.exists():
            raise FileNotFoundError(
                f"Cached file not found: {cache_path}. "
                f"Did you run the caching step?"
            )

        img = np.load(cache_path)  # (H, W, 6), uint8

        rgb = img[:, :, :3].astype(np.uint8)
        extra = img[:, :, 3:].astype(np.float32) / 255.0

        rgb_pil = Image.fromarray(rgb)
        if self.transform is not None:
            rgb_tensor = self.transform(rgb_pil)
        else:
            rgb_tensor = transforms.ToTensor()(rgb_pil)

        extra_tensor = torch.from_numpy(extra.transpose(2, 0, 1))
        extra_tensor = torch.nn.functional.interpolate(
            extra_tensor.unsqueeze(0),
            size=self.config.image_size,
            mode="bilinear",
            align_corners=False,
        ).squeeze(0)

        img_tensor = torch.cat([rgb_tensor, extra_tensor], dim=0)

        # If the row has targets, return them. If it's test data, return only img.
        has_targets = all(col in row for col in self.config.target_cols)
        if has_targets:
            targets = torch.tensor(
                [np.log1p(row[col]) for col in self.config.target_cols],
                dtype=torch.float32,
            )
            return img_tensor, targets
        else:
            return img_tensor


In [ ]:
def cache_images_for_split(df: pd.DataFrame, config: Config, cache_root: Path):
    """
    Save preprocessed arrays under:
      cache_root / (image_path + ".npy")
    """
    cache_root.mkdir(parents=True, exist_ok=True)

    for i, row in df.iterrows():
        rel_path = row["image_path"]  # 'train/ID....jpg' or 'test/ID....jpg'
        src_path = Path(config.input_dir) / rel_path
        dst_path = cache_root / (rel_path + ".npy")
        dst_path.parent.mkdir(parents=True, exist_ok=True)

        if dst_path.exists():
            continue

        img_bgr = cv2.imread(str(src_path))
        if img_bgr is None:
            print(f"[WARN] missing image: {src_path}")
            continue

        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_pre = ImagePreprocessor.preprocess(img_rgb, config)
        np.save(dst_path, img_pre)

        if (i + 1) % 500 == 0:
            print(f"  cached {i+1}/{len(df)} images...")


config = Config()
config.input_dir = str(DATA_ROOT)

train_csv = pd.read_csv(TRAIN_CSV)
print("Raw train.csv rows:", len(train_csv))

# pivot to 1 row per image
train_grouped = train_csv.pivot_table(
    index=[
        "image_path",
        "Sampling_Date",
        "State",
        "Species",
        "Pre_GSHH_NDVI",
        "Height_Ave_cm",
    ],
    columns="target_name",
    values="target",
    aggfunc="first",
).reset_index()
train_grouped = train_grouped.dropna(subset=config.target_cols)
print("Unique train images:", len(train_grouped))

test_csv = pd.read_csv(TEST_CSV)
print("Raw test.csv rows:", len(test_csv))
test_unique = test_csv[["image_path"]].drop_duplicates()
print("Unique test images:", len(test_unique))

print("\nCaching TRAIN images...")
cache_images_for_split(train_grouped, config, CACHE_ROOT)

print("\nCaching TEST images...")
cache_images_for_split(test_unique, config, CACHE_ROOT)

print("\n✅ Caching complete.")


In [ ]:
class BiomassLightningEffNet(LightningModule):
    """
    EfficientNetV2-S with warmup + full fine-tuning for CSIRO biomass.
    Stage 1: freeze backbone → train only head
    Stage 2: unfreeze backbone → lower LR
    """
    def __init__(
        self,
        num_channels: int,
        target_names: list[str],
        target_weights: dict[str, float],
        lr_head: float = 1e-4,
        lr_backbone: float = 1e-5,
        warmup_epochs: int = 2,
        pretrained: bool = True,
    ):
        super().__init__()
        self.save_hyperparameters()

        self.num_targets = len(target_names)
        self.target_names = target_names
        self.lr_head = lr_head
        self.lr_backbone = lr_backbone
        self.warmup_epochs = warmup_epochs

        self.register_buffer(
            "weights_tensor",
            torch.tensor([target_weights[t] for t in target_names], dtype=torch.float32),
        )

        self.backbone = timm.create_model(
            "tf_efficientnetv2_s",
            pretrained=pretrained,
            in_chans=num_channels,
            num_classes=0,
            global_pool="avg",
        )

        for p in self.backbone.parameters():
            p.requires_grad = False

        n_features = self.backbone.num_features
        self.head = nn.Sequential(
            nn.Linear(n_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, self.num_targets),
        )

        self.mae = MeanAbsoluteError()

    def unfreeze_backbone(self):
        print("\n🔓 Unfreezing EfficientNet backbone...\n")
        for p in self.backbone.parameters():
            p.requires_grad = True

    def forward(self, x):
        feats = self.backbone(x)
        return self.head(feats)

    def loss_fn(self, preds, targets):
        w = self.weights_tensor
        mse = (preds - targets) ** 2
        return (mse * w).mean()

    def weighted_r2(self, preds, targets):
        w = self.weights_tensor
        ss_res = torch.sum(w * (preds - targets) ** 2)
        y_mean = torch.sum(w * targets) / torch.sum(w)
        ss_tot = torch.sum(w * (targets - y_mean) ** 2)
        return 1.0 - ss_res / (ss_tot + 1e-8)

    def training_step(self, batch, batch_idx):
        imgs, targets = batch
        preds = self(imgs)
        loss = self.loss_fn(preds, targets)
        mae = self.mae(preds, targets)
        r2 = self.weighted_r2(preds, targets)
        self.log("train_loss", loss, prog_bar=True, on_epoch=True)
        self.log("train_mae", mae, on_epoch=True)
        self.log("train_r2", r2, prog_bar=True, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        imgs, targets = batch
        preds = self(imgs)
        loss = self.loss_fn(preds, targets)
        mae = self.mae(preds, targets)
        r2 = self.weighted_r2(preds, targets)
        self.log("val_loss", loss, prog_bar=True, on_epoch=True)
        self.log("val_mae", mae, on_epoch=True)
        self.log("val_r2", r2, prog_bar=True, on_epoch=True)
        return loss

    def configure_optimizers(self):
        params = [
            {"params": self.head.parameters(), "lr": self.lr_head},
            {"params": self.backbone.parameters(), "lr": self.lr_backbone},
        ]
        opt = torch.optim.AdamW(params)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=10)
        return {"optimizer": opt, "lr_scheduler": sched}

    def on_train_epoch_start(self):
        if self.current_epoch == self.warmup_epochs:
            self.unfreeze_backbone()


In [ ]:
config = Config()
config.input_dir = str(DATA_ROOT)

print("Using config:")
print("  targets:", config.target_cols)
print("  weights:", config.target_weights)
print("  channels:", config.total_input_channels)

# train_grouped already computed in the caching cell
df = train_grouped.sample(frac=1.0, random_state=42).reset_index(drop=True)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

fold_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(df), start=1):
    print("\n" + "="*70)
    print(f"🧪 Fold {fold}")
    print("="*70)

    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df   = df.iloc[val_idx].reset_index(drop=True)

    train_ds = CachedBiomassDataset(
        train_df, config, CACHE_ROOT, transform=get_transforms(config, is_train=True)
    )
    val_ds = CachedBiomassDataset(
        val_df, config, CACHE_ROOT, transform=get_transforms(config, is_train=False)
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=32,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=32,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
    )

    model = BiomassLightningEffNet(
        num_channels=config.total_input_channels,
        target_names=config.target_cols,
        target_weights=config.target_weights,
        lr_head=1e-4,
        lr_backbone=1e-5,
        warmup_epochs=2,
        pretrained=True,
    )

    logger = CSVLogger(save_dir=str(WORK_ROOT / "lightning_logs"),
                       name=f"fold_{fold}")

    ckpt_cb = ModelCheckpoint(
        dirpath=FOLD_DIR / f"fold_{fold}",
        filename="best",
        monitor="val_r2",
        mode="max",
        save_top_k=1,
    )
    es_cb = EarlyStopping(monitor="val_r2", mode="max", patience=5)
    lr_cb = LearningRateMonitor(logging_interval="epoch")

    trainer = Trainer(
        max_epochs=15,
        accelerator=device,
        devices=1,
        precision="16-mixed" if device == "cuda" else 32,
        logger=logger,
        callbacks=[ckpt_cb, es_cb, lr_cb],
        log_every_n_steps=1,
    )

    trainer.fit(model, train_loader, val_loader)
    best_score = ckpt_cb.best_model_score.item()
    fold_scores.append(best_score)
    print(f"✅ Fold {fold} best val R²: {best_score:.6f}")

print("\n🔥 5-FOLD CROSS VALIDATION COMPLETE")
for i, s in enumerate(fold_scores, start=1):
    print(f"Fold {i}: R² = {s:.6f}")
print("Avg R²:", np.mean(fold_scores), "Std:", np.std(fold_scores))


In [ ]:
@torch.no_grad()
def infer_and_build_submission():
    print("\n========================================================")
    print("🚀 Inference & submission generation")
    print("========================================================\n")

    raw_test = pd.read_csv(TEST_CSV)

    test_unique = raw_test[["image_path"]].drop_duplicates().reset_index(drop=True)
    print("Unique test images:", len(test_unique))

    config = Config()
    config.input_dir = str(DATA_ROOT)

    test_ds = CachedBiomassDataset(
        test_unique,
        config,
        CACHE_ROOT,
        transform=get_transforms(config, is_train=False),
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=32,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
    )

    # discover checkpoints
    fold_ckpts = []
    for fold in range(1, 6):
        path = FOLD_DIR / f"fold_{fold}" / "best.ckpt"
        if path.exists():
            fold_ckpts.append(path)
    if not fold_ckpts:
        raise FileNotFoundError("No fold checkpoints found in FOLD_DIR")

    print("\nFound fold checkpoints:")
    for p in fold_ckpts:
        print("  •", p)

    all_fold_preds = []
    target_order = [
        "Dry_Green_g",
        "Dry_Dead_g",
        "Dry_Clover_g",
        "GDM_g",
        "Dry_Total_g",
    ]

    for i, ckpt in enumerate(fold_ckpts, start=1):
        print(f"\nLoading Fold {i} model...")
        model = BiomassLightningEffNet.load_from_checkpoint(
            checkpoint_path=str(ckpt),
            map_location=device,
            strict=False,
        )
        model.to(device)
        model.eval()

        preds_list = []
        for batch in test_loader:
            imgs = batch.to(device)
            preds = model(imgs)
            preds_list.append(preds.cpu().numpy())

        preds_fold = np.concatenate(preds_list, axis=0)  # [N, 5]
        all_fold_preds.append(preds_fold)
        print(f"✔ Fold {i} predictions complete.")

    print("\nEnsembling folds...")
    all_fold_preds = np.stack(all_fold_preds, axis=0)   # [F, N, 5]
    ensemble_log = all_fold_preds.mean(axis=0)          # [N, 5]

    print("Applying inverse log1p...")
    ensemble = np.expm1(ensemble_log)                   # back to original scale

    # Build submission in sample_id,target format
    print("Building submission...")
    sub_rows = []

    for _, row in raw_test.iterrows():
        img_path = row["image_path"]
        tgt_name = row["target_name"]
        sample_id = row["sample_id"]

        img_idx = test_unique.index[test_unique["image_path"] == img_path][0]
        tgt_idx = target_order.index(tgt_name)
        value = ensemble[img_idx, tgt_idx]
        sub_rows.append([sample_id, value])

    submission = pd.DataFrame(sub_rows, columns=["sample_id", "target"])
    out_path = WORK_ROOT / "submission.csv"
    submission.to_csv(out_path, index=False)

    print("\n✅ Submission saved to:", out_path)
    display(submission.head())
    return submission

submission = infer_and_build_submission()
